### Data Ingestion

In [11]:
from langchain_core.documents import Document
from pathlib import Path

In [12]:
doc=Document(page_content="this page is RAG",
             metadata={"source":"example.txt",
                         "pages":1}
             )
doc

Document(metadata={'source': 'example.txt', 'pages': 1}, page_content='this page is RAG')

In [13]:
## create txt file
import os
os.makedirs("../data/text_files",exist_ok=True)

In [14]:
sample_text={
    "../data/text_files/python_intro.txt":"""I love Python, umm aaaaa ummm aaa
    Python is so cool and annoyinhg at the same time coz why do we have to remember so muc
    shit all the time, unlike java where the synatx is not much to remember
    uk what i dont like python at all """    
}
for file, content in sample_text.items():
    with open(file,'w',encoding="utf-8")as f:
        f.write(content)
print("Sample_text created")


Sample_text created


In [15]:
## text reading
from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document=loader.load()

def load_all_text_files(directory):
    all_docs = []

    text_files = list(Path(directory).glob("*.txt"))

    print(f"Found {len(text_files)} text files")

    for file in text_files:
        loader = TextLoader(str(file))
        docs = loader.load()

        for doc in docs:
            doc.metadata["source_file"] = file.name

        all_docs.extend(docs)

    return all_docs

In [38]:
documents0 = load_all_text_files("../data/text_files")
print(len(documents0))

Found 5 text files
5


In [39]:
from langchain_community.document_loaders import DirectoryLoader


dir_loader=DirectoryLoader("../data/text_files",
                           glob="**/*.txt",
                           loader_cls=TextLoader,
                           loader_kwargs={'encoding':'utf-8'},
                           show_progress=False)
doc=dir_loader.load()
doc


[Document(metadata={'source': '../data/text_files/java.txt'}, page_content='Java is an object-oriented programming language designed to be platform independent through the JVM.'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='I love Python, umm aaaaa ummm aaa\n    Python is so cool and annoyinhg at the same time coz why do we have to remember so muc\n    shit all the time, unlike java where the synatx is not much to remember\n    uk what i dont like python at all '),
 Document(metadata={'source': '../data/text_files/transformer.txt'}, page_content='Transformers rely on self-attention instead of recurrent networks. This allows them to process sequences in parallel.'),
 Document(metadata={'source': '../data/text_files/rag.txt'}, page_content='Retrieval Augmented Generation combines document retrieval with a language model. Retrieved documents are inserted into the prompt before generation.'),
 Document(metadata={'source': '../data/text_files/attentio

In [40]:
from langchain_community.document_loaders import (PyPDFLoader, PyMuPDFLoader)
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [42]:
def split_documents(doc,chunk_size=1000,chunk_overlap=200):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(doc)
    print(f"Split {len(doc)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [43]:
chunks = split_documents(documents0)
chunks

Split 5 documents into 5 chunks

Example chunk:
Content: Java is an object-oriented programming language designed to be platform independent through the JVM....
Metadata: {'source': '../data/text_files/java.txt', 'source_file': 'java.txt'}


[Document(metadata={'source': '../data/text_files/java.txt', 'source_file': 'java.txt'}, page_content='Java is an object-oriented programming language designed to be platform independent through the JVM.'),
 Document(metadata={'source': '../data/text_files/python_intro.txt', 'source_file': 'python_intro.txt'}, page_content='I love Python, umm aaaaa ummm aaa\n    Python is so cool and annoyinhg at the same time coz why do we have to remember so muc\n    shit all the time, unlike java where the synatx is not much to remember\n    uk what i dont like python at all'),
 Document(metadata={'source': '../data/text_files/transformer.txt', 'source_file': 'transformer.txt'}, page_content='Transformers rely on self-attention instead of recurrent networks. This allows them to process sequences in parallel.'),
 Document(metadata={'source': '../data/text_files/rag.txt', 'source_file': 'rag.txt'}, page_content='Retrieval Augmented Generation combines document retrieval with a language model. Retrieve

In [44]:
print(type(loader))
print(len(documents0))
print(type(documents0))
print(len(chunks))

<class 'langchain_community.document_loaders.text.TextLoader'>
5
<class 'list'>
5


## embedding

In [45]:
import numpy as np
from sentence_transformers import SentenceTransformer
import uuid
import chromadb
from chromadb.config import Settings
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [46]:
class EmbeddingManager:
    def __init__(self,modelname:str="all-MiniLM-L6-v2"):
        self.modelname=modelname
        self.model=None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model:{self.modelname}")
            self.model= SentenceTransformer(self.modelname)
            print(f"Model loaded successfully. Embedding dimension:{self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model{self.modelname}:{e}")
            raise

    def generate_embedding(self, texts:List[str])-> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings= self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape:{embeddings.shape}")
        return embeddings
        
em=EmbeddingManager()
em

Loading embedding model:all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5432.22it/s]


Model loaded successfully. Embedding dimension:384


In [47]:
class VectorStore:
    def __init__(self,collection_name:str="Text.documents", persist_directory:str="../data/vector_store"):
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialise_store()

    def _initialise_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"Text document embeddings for RAG"}
            )  
            print(f"Vectore store inialised. Collection:{self.collection_name}")
            print(f"Existing documents i collection:{self.collection.count()}")

        except Exception as e:
            print(f"Error initialisig vector store :{e}")
            raise

    def add_documents(self,documents0:List[Any],embeddings:np.ndarray):
        if len(documents0)!=len(embeddings):
            raise ValueError("both should be equal")
        print(f"Adding {len(documents0)} documents to vector store")
        ids=[]
        metadatas=[]
        doc_text=[]
        embbed_list=[]
        for i,(doc,embed) in enumerate(zip(documents0, embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)
            doc_text.append(doc.page_content)
            embbed_list.append(embed)

        try:
            self.collection.add(
                ids=ids,embeddings=embbed_list,metadatas=metadatas,
                documents=doc_text
            )
            print(f"Successfully added {len(documents0)} documents to vector store")
            print(f"Total documents i collection are:{self.collection.count()}")
        except Exception as e:
            print(f"Error adding doc:{e}")
            raise

vs=VectorStore()
vs

Vectore store inialised. Collection:Text.documents
Existing documents i collection:0


In [48]:
text=[doc.page_content for doc in chunks]

embeddings=em.generate_embedding(text)
vs.add_documents(chunks,embeddings)

Generating embeddings for 5 texts...


Batches: 100%|██████████| 1/1 [00:03<00:00,  3.45s/it]

Generated embeddings with shape:(5, 384)
Adding 5 documents to vector store
Successfully added 5 documents to vector store
Total documents i collection are:5


RAG Retrieval

In [49]:
class RAGRetriever:
    def __init__(self,vs:VectorStore,em:EmbeddingManager):
        self.vs=vs
        self.em=em

    def retrieve(self,query:str, top_k:int=5, score_threshold:float=0.0) -> List[Dict[str,Any]]:
        print(f"retrieval documents for query: '{query}")
        print(f"Top K:{top_k}, score threshold:{score_threshold}")
        query_embedding=self.em.generate_embedding([query])[0]

        try:
            results=self.vs.collection.query(
                query_embeddings=[query_embedding],
                n_results=top_k
            )
            retrieved_doc=[]
            if results['documents'] and results['documents'][0]:
                doc=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]

                for i,(doc_id, doc, metadata,dist) in enumerate(zip(ids,doc,metadatas,distances)):
                    similarity_score=1-dist

                    if similarity_score>=score_threshold:
                        retrieved_doc.append({
                            'id':doc_id,
                            'content':doc,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':dist,
                            'rank':i+1
                        })
                print(f"Retrieved {len(retrieved_doc)} documents (after filtering)")

            else:
                print("no docs found")
            return retrieved_doc
        except Exception as e:
            print(f"Error during retrieval:{e}")
            return []
        
ragr=RAGRetriever(vs,em)

In [50]:
ragr.retrieve("What is relation between attention and transformer")

retrieval documents for query: 'What is relation between attention and transformer
Top K:5, score threshold:0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  8.06it/s]

Generated embeddings with shape:(1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_5fd791fa_2',
  'content': 'Transformers rely on self-attention instead of recurrent networks. This allows them to process sequences in parallel.',
  'metadata': {'source': '../data/text_files/transformer.txt',
   'source_file': 'transformer.txt',
   'content_length': 117,
   'doc_index': 2},
  'similarity_score': 0.20064347982406616,
  'distance': 0.7993565201759338,
  'rank': 1},
 {'id': 'doc_673d109f_4',
  'content': 'The attention mechanism helps a model determine which words are most important when processing a sentence. Attention mechanisms allow neural networks to focus on the most relevant parts of an input sequence. Each token receives a weight indicating its importance.',
  'metadata': {'source_file': 'attention.txt',
   'content_length': 263,
   'source': '../data/text_files/attention.txt',
   'doc_index': 4},
  'similarity_score': 0.09086024761199951,
  'distance': 0.9091397523880005,
  'rank': 2}]

## Intergration with llm

In [51]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
load_dotenv()
GEMINI_API_KEY=os.getenv("GEMINI_API_KEY")
llm=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",temperature=0.1)

def rag_simple(query, retriever, llm,top_k=3):
    results=retriever.retrieve(query,top_k=top_k)
    context="\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found"
    
    prompt=f"""Use the following context to answer the question
        Context:{context}
        Question:{query}
        Answer:"""
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [52]:
answer=rag_simple("What is attention mechanism?",ragr,llm)
print(answer)

retrieval documents for query: 'What is attention mechanism?
Top K:3, score threshold:0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.46it/s]


Generated embeddings with shape:(1, 384)
Retrieved 1 documents (after filtering)
The attention mechanism is a feature that helps a model determine which words are most important when processing a sentence. It allows neural networks to focus on the most relevant parts of an input sequence, with each token receiving a weight indicating its importance.


In [53]:
query_embedding = em.generate_embedding(
    ["What is attention"]
)[0]

results = vs.collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print(results)

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.35it/s]

Generated embeddings with shape:(1, 384)
{'ids': [['doc_673d109f_4', 'doc_5fd791fa_2', 'doc_110bf1e1_3']], 'embeddings': None, 'documents': [['The attention mechanism helps a model determine which words are most important when processing a sentence. Attention mechanisms allow neural networks to focus on the most relevant parts of an input sequence. Each token receives a weight indicating its importance.', 'Transformers rely on self-attention instead of recurrent networks. This allows them to process sequences in parallel.', 'Retrieval Augmented Generation combines document retrieval with a language model. Retrieved documents are inserted into the prompt before generation.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'doc_index': 4, 'source_file': 'attention.txt', 'content_length': 263, 'source': '../data/text_files/attention.txt'}, {'source': '../data/text_files/transformer.txt', 'doc_index': 2, 'source_file': 'transformer.txt', '